In [2]:
# устанавливаем необходимые библиотеки 

%pip install opensearch-py==2.8.0
%pip install pandas==2.0.3


Defaulting to user installation because normal site-packages is not writeable
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
botocore 1.23.24 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.3.0 which is incompatible.
cloud-ml 0.0.1 requires requests<=2.28.1,>=2.22.0, but you have requests 2.32.3 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 62.4 MB/s eta 0:00:0000:01

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [9]:
%mkdir -p ~/.opensearch && \
wget "https://storage.yandexcloud.net/cloud-certs/CA.pem" \
     --output-document ~/.opensearch/root.crt && \
chmod 0600 ~/.opensearch/root.crt

--2025-02-11 11:12:46--  https://storage.yandexcloud.net/cloud-certs/CA.pem
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3579 (3.5K) [application/x-x509-ca-cert]
Saving to: ‘/home/jupyter/.opensearch/root.crt’

     0K ...                                                   100% 5.16G=0s

2025-02-11 11:12:46 (5.16 GB/s) - ‘/home/jupyter/.opensearch/root.crt’ saved [3579/3579]



In [73]:
import pandas as pd


from opensearchpy import OpenSearch, helpers

simple_index_name = "simple-index-webinar"
lemmer_index = "lemmer-index-webinar"

CA = "~/.opensearch/root.crt"
PASS = "pass"
HOSTS = ["host"]

conn = OpenSearch(
    HOSTS, http_auth=("admin", PASS), use_ssl=True, verify_certs=True, ca_certs=CA
)

conn.lemmer_index = lemmer_index
conn.no_lemmer_index = simple_index_name


In [ ]:
# базовый метод для создания индексов

In [74]:
def create_index(conn: OpenSearch, index_name:str, index_body: dict):
    return conn.indices.create(index=index_name, body=index_body)

In [ ]:
# первый кейс - для базовой демонстрации

In [79]:
# Документ для индексации

index_name = "simple-index-short"

document = {"book": "Ночь, когда шел дождь", "author": "Юджиния Райли"}

create_index(conn=conn, index_name=index_name, index_body=None)

conn.index(index_name, body=document, refresh=True)

{'_index': 'simple-index-short',
 '_id': 'ICPu-ZQBHoMVQdjYYbnc',
 '_version': 1,
 'result': 'created',
 'forced_refresh': True,
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 0,
 '_primary_term': 1}

In [80]:
# обычный запрос
standard_search = conn.search(
    index="simple-index-short",
    body={
        "query": {
            "match": {
                "book": {
                    "query": "идут дожди"
                }
            }
        }
    }
)
print("Standard search results:", standard_search['hits']['hits'])

Standard search results: []


In [81]:
# Документ для индексации
index_name = "lemmer-index-short"

document = {"book": "Ночь, когда шел дождь", "author": "Юджиния Райли"}
index_body = {
    "settings": {
        "analysis": {
            "analyzer": {
                "lemmer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["yandex_lemmer"],
                }
            }
        }
    },
    "mappings": {
        "properties": {"book": {"type": "text", "analyzer": "lemmer"}}
    },
    }
create_index(conn=conn, index_name=index_name, index_body=index_body)

conn.index(index_name, body=document)

{'_index': 'lemmer-index-short',
 '_id': 'ISPu-ZQBHoMVQdjYfLls',
 '_version': 1,
 'result': 'created',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 0,
 '_primary_term': 1}

In [82]:
lemmer_search = conn.search(
    index="lemmer-index-short",
    body={
        "query": {
            "match": {
                "book": {
                    "query": "идут дожди",
                    "analyzer": "lemmer"
                }
            }
        }
    }
)


print("Lemmer search results:", lemmer_search['hits']['hits'])


Lemmer search results: [{'_index': 'lemmer-index-short', '_id': 'ISPu-ZQBHoMVQdjYfLls', '_score': 0.6662112, '_source': {'book': 'Ночь, когда шел дождь', 'author': 'Юджиния Райли'}}]


In [ ]:
# переходим к кейсу с geo reviews

In [19]:
!curl -L -o yandex-geo-reviews-dataset-2023.zip  https://www.kaggle.com/api/v1/datasets/download/kyakovlev/yandex-geo-reviews-dataset-2023

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  199M  100  199M    0     0  23.3M      0  0:00:08  0:00:08 --:--:-- 29.5M


In [75]:
# описываем структуру датафрейма гео данных

columns = ['address', 'name_ru', 'rating', 'rubrics', 'text']


dtypes = {
    'address': str,
    'name_ru': str,
    'rating': str,
    'rubrics': str,
    'text': str
}



In [ ]:
# подготавливаем данные и вставляем в БД без леммера

In [76]:
df = pd.read_csv(
"./geo-reviews-dataset-2023.csv",
names=columns,  
dtype=dtypes,   
header=None     
)

# избавляемся от пустых значений
df = df.fillna('empty')
# конвертируем rating во float
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
# добавляем колонку с названием индекса в opensearch
df['_index'] = simple_index_name
# добавляем колонку с порядковым номером записи
df['_id'] = df.index

index_body = {
        "mappings": {
            "properties": {
                "address": {"type": "text"},
                "name_ru": {"type": "text", },
                "rubrics": {"type": "text"},
                "rating": {"type": "float"},
                "text": {"type": "text"},
            }
        },
    }

create_index(conn=conn, index_name=simple_index_name, index_body=index_body)

helpers.bulk(conn, df.to_dict('records')[1:])

(500000, [])

In [ ]:
# подготавливаем данные и вставляем в БД с леммером

In [78]:
df = pd.read_csv(
"./geo-reviews-dataset-2023.csv",
names=columns,  
dtype=dtypes,   
header=None     
)

# избавляемся от пустых значений
df = df.fillna('empty')
# конвертируем rating во float
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
# добавляем колонку с названием индекса в opensearch
df['_index'] = lemmer_index
# добавляем колонку с порядковым номером записи
df['_id'] = df.index

index_body = {
    "settings": {
        "analysis": {
            "analyzer": {
                "lemmer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["yandex_lemmer"],
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "address": {"type": "text"},
            "name_ru": {"type": "text"},
            "rubrics": {"type": "text"},
            "rating": {"type": "float"},
            "text": {"type": "text", "analyzer": "lemmer"},
        }
    },
    }

create_index(conn=conn, index_name=lemmer_index, index_body=index_body)

helpers.bulk(conn, df.to_dict('records')[1:])

(500000, [])

In [ ]:
# метод для сравнения

In [83]:
def compare_lemmatization_effectiveness(conn, lemmer_index, no_lemmer_index):
    test_queries = [
        {"base_word": "автомобилист", "variations": ["автомобилистов", "автомобилист"]},
        {"base_word": "половинный", "variations": ["половинными", "половинная"]},
        {"base_word": "убегать", "variations": ["убегающему", "убегать"]},
    ]

    results = {}

    for test_case in test_queries:
        print(f"\n{'='*80}")
        print(f"Анализ слова: {test_case['base_word']}")

        for variation in test_case["variations"]:
            # Query for counts
            count_query = {
                "query": {"match_phrase": {"text": variation}},
                "track_total_hits": True,
            }

            # Query for examples
            example_query = {
                "query": {"match_phrase": {"text": variation}},
                "size": 2,
                "highlight": {
                    "fields": {"text": {}},
                    "pre_tags": ["**"],
                    "post_tags": ["**"],
                },
            }

            # Get counts
            lemmer_result = conn.search(index=lemmer_index, body=count_query, size=0)
            no_lemmer_result = conn.search(
                index=no_lemmer_index, body=count_query, size=0
            )

            # Get examples
            lemmer_examples = conn.search(index=lemmer_index, body=example_query)
            no_lemmer_examples = conn.search(index=no_lemmer_index, body=example_query)

            results[variation] = {
                "lemmer_hits": lemmer_result["hits"]["total"]["value"],
                "no_lemmer_hits": no_lemmer_result["hits"]["total"]["value"],
            }

            print(f"\n{'-'*80}")
            print(f"Форма слова: {variation}")
            print(
                f"С лемматизацией: {lemmer_result['hits']['total']['value']} совпадений"
            )
            print(
                f"Без лемматизации: {no_lemmer_result['hits']['total']['value']} совпадений"
            )

            print("\nПримеры с лемматизацией:")
            for hit in lemmer_examples["hits"]["hits"]:
                highlight = hit.get("highlight", {}).get(
                    "text", [hit["_source"]["text"][:200]]
                )[0]
                print(f"- {highlight}...")

            print("\nПримеры без лемматизации:")
            for hit in no_lemmer_examples["hits"]["hits"]:
                highlight = hit.get("highlight", {}).get(
                    "text", [hit["_source"]["text"][:200]]
                )[0]
                print(f"- {highlight}...")

    return results


In [85]:
print(compare_lemmatization_effectiveness(conn, lemmer_index, simple_index_name))


Анализ слова: автомобилист

--------------------------------------------------------------------------------
Форма слова: автомобилистов
С лемматизацией: 133 совпадений
Без лемматизации: 64 совпадений

Примеры с лемматизацией:
- Супер магазин для **автомобилистов**. Рекомендую....
- Любимое место перекуса дальнобойщиков и гостей-**автомобилистов**. Чисто, уютно, вкусно...

Примеры без лемматизации:
- Супер магазин для **автомобилистов**. Рекомендую....
- Любимое место перекуса дальнобойщиков и гостей-**автомобилистов**. Чисто, уютно, вкусно...

--------------------------------------------------------------------------------
Форма слова: автомобилист
С лемматизацией: 133 совпадений
Без лемматизации: 16 совпадений

Примеры с лемматизацией:
- Супер магазин для **автомобилистов**. Рекомендую....
- Любимое место перекуса дальнобойщиков и гостей-**автомобилистов**. Чисто, уютно, вкусно...

Примеры без лемматизации:
- Занималась в автошколе «**Автомобилист**»!...
- Вспомнился пионер лагерь *